In [28]:
import os
import glob
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep
from matplotlib.backends.backend_pdf import PdfPages
from coffea.util import load

from btag_norm import apply_btag_norm   # same normalization as the datacard pipeline

# ==========================================
# 1. GLOBAL SETTINGS & CONFIGURATION
# ==========================================
hep.style.use(hep.style.CMS)

# ---------------- USER SWITCHES ----------------
# One of: "2016_PreVFP", "2016_PostVFP", "2016" (= PreVFP + PostVFP combined), "2017", "2018"
TARGET_ERA = "2018"

# Xbb (ZH_bbvLscore) cut: None (no cut), "LP", "MP" or "HP".
# Thresholds are applied per sub-era, so the combined "2016" uses the
# PreVFP thresholds on PreVFP events and the PostVFP thresholds on PostVFP events.
XBB_WP = None

# WP used by analysis_cuts() in the non-bb signal check (keep in sync with the datacard script)
ANALYSIS_XBB_WP = "LP"

# b-tag shape SF normalization: one or more JSON files from btag_norm.py
# (e.g. the signal, background and minor-background files). Merged in order.
BTAG_NORM_FILES = ["btag_calib/btag_norm_sig.json", "btag_calib/btag_norm_bkg.json", "btag_calib/btag_norm_minor.json"]
# -----------------------------------------------

# Input directory per sub-era  <-- UPDATE THESE PATHS
# If PreVFP and PostVFP live in the same directory, just point both entries to it
# (duplicate files are removed automatically).
COFFEA_DIR_DICT = {
    "2016_PreVFP":  "Nominal2016APV",
    "2016_PostVFP": "Nominal2016",
    "2017":         "Nominal2017",
    "2018":         "Nominal2018",
}

LUMI_DICT = {  # in fb^-1
    "2016_PreVFP":  19.5,
    "2016_PostVFP": 16.8,
    "2016":         round(19.5 + 16.8, 1),
    "2017":         41.5,
    "2018":         59.8,
}

XBB_WP_DICT = {
    "2016_PreVFP":  {"LP": 0.9088, "MP": 0.9737, "HP": 0.9883},
    "2016_PostVFP": {"LP": 0.9137, "MP": 0.9735, "HP": 0.9883},
    "2017":         {"LP": 0.9105, "MP": 0.9714, "HP": 0.9870},
    "2018":         {"LP": 0.9172, "MP": 0.9734, "HP": 0.9880},
}

# Which sub-eras make up each target era
ERA_COMPONENTS = {
    "2016_PreVFP":  ["2016_PreVFP"],
    "2016_PostVFP": ["2016_PostVFP"],
    "2016":         ["2016_PreVFP", "2016_PostVFP"],
    "2017":         ["2017"],
    "2018":         ["2018"],
}

# --- Validate switches & derive era-dependent settings ---
if TARGET_ERA not in LUMI_DICT:
    raise ValueError(f"Unknown TARGET_ERA '{TARGET_ERA}'. Choose from {list(LUMI_DICT)}")
for wp_name, wp in [('XBB_WP', XBB_WP), ('ANALYSIS_XBB_WP', ANALYSIS_XBB_WP)]:
    if wp not in (None, "LP", "MP", "HP"):
        raise ValueError(f"Unknown {wp_name} '{wp}'. Choose None, 'LP', 'MP' or 'HP'")

LUMI = LUMI_DICT[TARGET_ERA]
TARGET_SUB_ERAS = ERA_COMPONENTS[TARGET_ERA]
COFFEA_DIR = [COFFEA_DIR_DICT[e] for e in TARGET_SUB_ERAS]

XBB_TAG = f"Xbb{XBB_WP}" if XBB_WP else "NoXbb"
OUTPUT_TAG = f"{TARGET_ERA}_{XBB_TAG}"


def load_btag_norm(paths):
    """Merge the b-tag normalization tables from btag_norm.py (later files win)."""
    merged = None
    for path in paths:
        if not os.path.exists(path):
            print(f"🚨 WARNING: b-tag normalization file '{path}' not found; skipping it.")
            continue
        table = json.load(open(path))
        if merged is None:
            merged = table
            continue
        if table.get('njet_edges') != merged.get('njet_edges'):
            raise ValueError(f"{path}: njet_edges differ from the other b-tag normalization files")
        for era, samples in table.items():
            if era != 'njet_edges':
                merged.setdefault(era, {}).update(samples)
    return merged


BTAG_NORM = load_btag_norm(BTAG_NORM_FILES)

print("=" * 50)
print(f" Era        : {TARGET_ERA}  (sub-eras: {', '.join(TARGET_SUB_ERAS)})")
print(f" Lumi       : {LUMI} fb^-1")
print(f" Input dirs : {', '.join(dict.fromkeys(COFFEA_DIR))}")
if XBB_WP:
    thr_str = ", ".join(f"{e}: > {XBB_WP_DICT[e][XBB_WP]}" for e in TARGET_SUB_ERAS)
    print(f" Xbb cut    : {XBB_WP} ({thr_str})")
else:
    print(f" Xbb cut    : none")
print(f" b-tag norm : {', '.join(BTAG_NORM_FILES) if BTAG_NORM is not None else 'NONE (b-tag SFs not normalized!)'}")
print(f" Output tag : {OUTPUT_TAG}")
print("=" * 50)


def get_dataset_era(dataset):
    """Map a dataset name to its sub-era. Returns None if it can't be determined."""
    d = dataset.lower()
    if any(k in d for k in ['apv', 'prevfp', 'hipm']):
        return '2016_PreVFP'
    if '2016' in d or 'postvfp' in d:
        return '2016_PostVFP'
    if '2017' in d:
        return '2017'
    if '2018' in d:
        return '2018'
    return None


def xbb_threshold(df_, wp):
    """Per-event Xbb threshold, looked up from each event's sub-era."""
    thr_map = {era: XBB_WP_DICT[era][wp] for era in XBB_WP_DICT}
    return df_['era'].astype(str).map(thr_map).astype(float)


def cuts(df_):
    base = (
        (df_['n_ak4jets']   >= 5)       &
        (df_['n_b_outZH']   >= 2)       &
        (df_['ZH_pt']      >= 200) &
        (df_['ZH_M']        >= 50)      &
        (df_['ZH_M']        <= 200)
    )
    if XBB_WP is not None:
        base &= (df_['ZH_bbvLscore'] > xbb_threshold(df_, XBB_WP))
    return base


XSEC_DICT = {
    "TTZToLLNuNu": 0.2529, "ttHTobb": 0.2934, "ttHToNonbb": 0.215,
    "TTZToQQ": 0.5297, "TTZToBB": 0.1157, "TTToHadronic": 380.095,
    "TTToSemiLeptonic": 364.018, "TTTo2L2Nu": 88.29, "TTbb_Hadronic": 9.286,
    "TTbb_SemiLeptonic": 8.168, "TTbb_2L2Nu": 1.807, "TTWJetsToLNu": 0.0197,
    "TTWJetsToQQ": 0.0019, "TTTT": 0.0394, "TTGJets": 0.0092,
    "ST_tW_top": 11.4, "ST_tW_antitop": 11.5, "ST_s_channel": 22.6,
    "ST_t_channel_top": 224.8, "ST_t_channel_antitop": 120.9,
    "QCD_HT500to700": 29980.0, "QCD_HT700to1000": 6334.0, "QCD_HT1000to1500": 1088.0,
    "QCD_HT1500to2000": 99.11, "QCD_HT2000toInf": 20.23,
    "WJetsToLNu_HT-70To100": 1524.0, "WJetsToLNu_HT-100To200": 1256.0,
    "WJetsToLNu_HT-200To400": 335.5, "WJetsToLNu_HT-400To600": 45.25,
    "WJetsToLNu_HT-600To800": 10.97, "WJetsToLNu_HT-800To1200": 4.933,
    "WJetsToLNu_HT-1200To2500": 1.074, "WJetsToLNu_HT-2500ToInf": 0.03216,
    "DYJetsToLL_HT-70to100": 175.3, "DYJetsToLL_HT-100to200": 139.4,
    "DYJetsToLL_HT-200to400": 38.4, "DYJetsToLL_HT-400to600": 5.174,
    "DYJetsToLL_HT-600to800": 1.258, "DYJetsToLL_HT-800to1200": 0.5765,
    "DYJetsToLL_HT-1200to2500": 0.1331, "DYJetsToLL_HT-2500toInf": 0.002987
}

# --- Processes ---
bkg_processes = ['VJets', 'QCD', 'tt_B', 'TTBar', 'SingleTop', 'TTX'] 
sig_processes = ['ttZ', 'ttH']
data_process = 'data_obs'
all_processes = bkg_processes + sig_processes + [data_process]

# --- Systematics ---
syst_bases = [
    #'AK4PFPuppi_JER', 'AK4PFPuppi_JES_AbsoluteMPFBias', 'AK4PFPuppi_JES_AbsoluteScale',
    # 'AK4PFPuppi_JES_AbsoluteStat', 'AK4PFPuppi_JES_FlavorQCD', 'AK4PFPuppi_JES_Fragmentation',
    # 'AK4PFPuppi_JES_PileUpDataMC', 'AK4PFPuppi_JES_PileUpEnvelope', 'AK4PFPuppi_JES_PileUpMuZero',
    # 'AK4PFPuppi_JES_PileUpPtBB', 'AK4PFPuppi_JES_PileUpPtEC1', 'AK4PFPuppi_JES_PileUpPtEC2',
    # 'AK4PFPuppi_JES_PileUpPtHF', 'AK4PFPuppi_JES_PileUpPtRef', 'AK4PFPuppi_JES_RelativeBal',
    # 'AK4PFPuppi_JES_RelativeFSR', 'AK4PFPuppi_JES_RelativeJEREC1', 'AK4PFPuppi_JES_RelativeJEREC2',
    # 'AK4PFPuppi_JES_RelativeJERHF', 'AK4PFPuppi_JES_RelativePtBB', 'AK4PFPuppi_JES_RelativePtEC1',
    # 'AK4PFPuppi_JES_RelativePtEC2', 'AK4PFPuppi_JES_RelativePtHF', 'AK4PFPuppi_JES_RelativeSample',
    # 'AK4PFPuppi_JES_RelativeStatEC', 'AK4PFPuppi_JES_RelativeStatFSR', 'AK4PFPuppi_JES_RelativeStatHF',
    # 'AK4PFPuppi_JES_SinglePionECAL', 'AK4PFPuppi_JES_SinglePionHCAL', 'AK4PFPuppi_JES_TimePtEta',
    # 'AK8PFPuppi_JER', 'AK8PFPuppi_JES_AbsoluteMPFBias', 'AK8PFPuppi_JES_AbsoluteScale',
    # 'AK8PFPuppi_JES_AbsoluteStat', 'AK8PFPuppi_JES_FlavorQCD', 'AK8PFPuppi_JES_Fragmentation',
    # 'AK8PFPuppi_JES_PileUpDataMC', 'AK8PFPuppi_JES_PileUpEnvelope', 'AK8PFPuppi_JES_PileUpMuZero',
    # 'AK8PFPuppi_JES_PileUpPtBB', 'AK8PFPuppi_JES_PileUpPtEC1', 'AK8PFPuppi_JES_PileUpPtEC2',
    # 'AK8PFPuppi_JES_PileUpPtHF', 'AK8PFPuppi_JES_PileUpPtRef', 'AK8PFPuppi_JES_RelativeBal',
    # 'AK8PFPuppi_JES_RelativeFSR', 'AK8PFPuppi_JES_RelativeJEREC1', 'AK8PFPuppi_JES_RelativeJEREC2',
    # 'AK8PFPuppi_JES_RelativeJERHF', 'AK8PFPuppi_JES_RelativePtBB', 'AK8PFPuppi_JES_RelativePtEC1',
    # 'AK8PFPuppi_JES_RelativePtEC2', 'AK8PFPuppi_JES_RelativePtHF', 'AK8PFPuppi_JES_RelativeSample',
    # 'AK8PFPuppi_JES_RelativeStatEC', 'AK8PFPuppi_JES_RelativeStatFSR', 'AK8PFPuppi_JES_RelativeStatHF',
    # 'AK8PFPuppi_JES_SinglePionECAL', 'AK8PFPuppi_JES_SinglePionHCAL', 'AK8PFPuppi_JES_TimePtEta',
    #'ele_scale', 'ele_smear', 'unclust_En', 'muon_scale', 'muon_smear'
]

# b-tag shape SF sources (columns btag_sf<src>_up / btag_sf<src>_down)
BTAG_SOURCES = ['hf', 'lf', 'cferr1', 'cferr2', 'hfstats1', 'hfstats2', 'lfstats1', 'lfstats2']

weight_vars = [
    'genWeight', 'norm_weight',
    'topptWeight', 'topptWeight_Up', 'topptWeight_Down',
    'ele_reco_sf', 'ele_reco_sfup', 'ele_reco_sfdown',
    'ele_id_sf', 'ele_id_sfup', 'ele_id_sfdown',
    'ele_trig_sf', 'ele_trig_sfup', 'ele_trig_sfdown',
    'mu_id_sf', 'mu_id_sfup', 'mu_id_sfdown',
    'mu_iso_sf', 'mu_iso_sfup', 'mu_iso_sfdown',
    'mu_trig_sf', 'mu_trig_sfup', 'mu_trig_sfdown',
    'mu_reco_sf', 'mu_reco_sfup', 'mu_reco_sfdown',
    'puWeight', 'puWeight_up', 'puWeight_down',
    'isr_up', 'isr_down', 'fsr_up', 'fsr_down',
    'mu_r_up', 'mu_r_down', 'mu_f_up', 'mu_f_down',
    'mu_rf_up', 'mu_rf_down',
    'btag_sf',
    'xbb_sf', 'xbb_sfup', 'xbb_sfdown'
] + [f'btag_sf{src}_{d}' for src in BTAG_SOURCES for d in ('up', 'down')]

weight_syst_mapping = {
    'topptWeight': ('topptWeight', 'topptWeight_Up', 'topptWeight_Down'),
    'xbb_sf': ('xbb_sf', 'xbb_sfup', 'xbb_sfdown'),
    'ele_reco_sf': ('ele_reco_sf', 'ele_reco_sfup', 'ele_reco_sfdown'),
    'ele_id_sf':   ('ele_id_sf', 'ele_id_sfup', 'ele_id_sfdown'),
    'ele_trig_sf':   ('ele_trig_sf', 'ele_trig_sfup', 'ele_trig_sfdown'),
    'mu_id_sf':    ('mu_id_sf', 'mu_id_sfup', 'mu_id_sfdown'),
    'mu_iso_sf':   ('mu_iso_sf', 'mu_iso_sfup', 'mu_iso_sfdown'),
    'mu_trig_sf':  ('mu_trig_sf', 'mu_trig_sfup', 'mu_trig_sfdown'),
    'mu_reco_sf':  ('mu_reco_sf', 'mu_reco_sfup', 'mu_reco_sfdown'),
    'puWeight':    ('puWeight', 'puWeight_up', 'puWeight_down'),
    'isr':         (None, 'isr_up', 'isr_down'), 
    'fsr':         (None, 'fsr_up', 'fsr_down'),
    'mu_r':        (None, 'mu_r_up', 'mu_r_down'),
    'mu_f':        (None, 'mu_f_up', 'mu_f_down'),
    'mu_rf':       (None, 'mu_rf_up', 'mu_rf_down')
}
# One entry per b-tag source: the varied SF replaces the nominal (normalized) btag_sf
weight_syst_mapping.update({f'btag_{src}': ('btag_sf', f'btag_sf{src}_up', f'btag_sf{src}_down')
                            for src in BTAG_SOURCES})

systematics = ['nominal'] #+ [f"{s}Up" for s in syst_bases] + [f"{s}Down" for s in syst_bases]

validation_vars = [
    'nPV',# 'nPVGood',
    'MET_pt', 'MET_phi', 'lep_pt', 'ele_pt', 'muon_pt',
    'lep_eta', 'ele_eta', 'muon_eta', 'n_ak4', 'n_bjet', 'n_ak8',
    'jet1_pt', 'jet2_pt', 'bjet1_pt', #'bjet2_pt', 
    'jet1_eta', 'jet2_eta',
    'bjet1_eta', #'bjet2_eta', 
    'jet1_btag', 'jet2_btag', 'bjet1_btag', #'bjet2_btag',
    'fatjet1_pt', 'fatjet1_eta', 'fatjet1_mass', 'ZH_bbvLscore'
]

#validation_vars = ['MET_phi', 'lep_pt', 'lep_eta', 'ZH_bbvLscore']

cut_vars = ['ZH_bbvLscore', 'ZH_M', 'ZH_pt', 'n_b_outZH', 'n_ak4jets', 'MET_pt']
vars_to_extract = validation_vars + weight_vars + cut_vars

binning_dict = {
    'nPV': (50, 0, 70), #'nPVGood': (50, 0, 70),
    'MET_pt': (40, 0, 500), 'MET_phi': (30, -3.14, 3.14),
    'lep_pt': (40, 0, 500), 'ele_pt': (40, 0, 500), 'muon_pt': (40, 0, 500),
    'lep_eta': (30, -2.5, 2.5), 'ele_eta': (30, -2.5, 2.5), 'muon_eta': (30, -2.4, 2.4),
    'n_ak4': (15, 0, 15), 'n_bjet': (10, 0, 10), 'n_ak8': (5, 0, 5),
    'jet1_pt': (40, 0, 500), 'jet2_pt': (40, 0, 500),
    'bjet1_pt': (40, 0, 500), 'bjet2_pt': (40, 0, 500),
    'jet1_eta': (30, -2.5, 2.5), 'jet2_eta': (30, -2.5, 2.5),
    'bjet1_eta': (30, -2.5, 2.5), 'bjet2_eta': (30, -2.5, 2.5),
    'jet1_btag': (20, 0, 1), 'jet2_btag': (20, 0, 1),
    'bjet1_btag': (20, 0, 1), 'bjet2_btag': (20, 0, 1),
    'fatjet1_pt': (40, 200, 500), 'fatjet1_eta': (30, -2.5, 2.5), 'fatjet1_mass': (40, 0, 200),
    'ZH_bbvLscore': (30, 0, 1)
}
default_binning = (40, 0, 500)

bkg_colors = {'VJets':'#3f90da', 'QCD':'#ffa90e', 'tt_B':'#bd1f01', 'TTBar':'#94a4a2', 'SingleTop':'#e76300', 'TTX':'#b9ac70'}
sig_colors = {'ttZ':'#832db6', 'ttH':'#a96b59'}
mc_colors = {**bkg_colors, **sig_colors}

process_labels = {
    'VJets': 'V+Jets', 'QCD': 'QCD', 'tt_B': r'$t\bar{t}+bb$',
    'TTBar': r'$t\bar{t} + lf, t\bar{t}+cc$', 'SingleTop': 'single top',
    'TTX': r'$t\bar{t}t\bar{t}, t\bar{t}W, t\bar{t}HW, t\bar{t}Hq$',
    'ttZ': r'$t\bar{t}Z$', 'ttH': r'$t\bar{t}H$', 'data_obs': 'Data', 'QCD': 'QCD', 'VV': 'VV'
}

# ==========================================
# 2. DATA EXTRACTION & WEIGHTING FUNCTIONS
# ==========================================

def getZhbbWeight(df_, year=2018):
    if 'norm_weight' not in df_.columns:
        return pd.Series(1.0, index=df_.index) 

    weight = df_['norm_weight'].copy()
    
    gen_w = df_.get('genWeight', pd.Series(1.0, index=df_.index)).fillna(1.0)
    weight *= np.sign(gen_w)
    
    sfs = ['ele_reco_sf', 'ele_id_sf', 'mu_id_sf', 'mu_iso_sf', 'mu_trig_sf', 'puWeight', 'topptWeight', 'ele_trig_sf', 'l1prefire_sf', 'mu_reco_sf', 'xbb_sf', 'btag_sf']  
    
    for sf in sfs:
        sf_col = df_.get(sf, pd.Series(1.0, index=df_.index)).fillna(1.0)
        weight *= sf_col
        
    return weight
    
def remove_ttzqq_zbb(df, dataset):
    """Drop Z->bb events from TTZToQQ (the 'old_ttzbb' events); TTZToBB covers them."""
    if 'ttztoqq' not in dataset.lower():
        return df
    if 'Zbb' not in df.columns:
        print(f"  -> [WARNING] events_Zbb missing for {dataset}; Z->bb overlap NOT removed!")
        return df
    n_before = len(df)
    df = df[~df['Zbb'].astype(bool)]
    print(f"  -> {dataset}: removed {n_before - len(df):,} of {n_before:,} Z->bb events")
    return df

# ==========================================
# NON-BB SIGNAL CHECK
# ==========================================
SIGNAL_TRUTH_FLAG = {'ttH': 'Hbb', 'ttZ': 'Zbb'}   # read from events_Hbb / events_Zbb

def analysis_cuts(df_):
    """Datacard selection (applied on top of the looser validation cuts).
    The Xbb threshold is taken per sub-era from XBB_WP_DICT[era][ANALYSIS_XBB_WP]."""
    return (
        (df_['n_ak4jets'] >= 5) & (df_['n_b_outZH'] == 2) &
        (df_['ZH_bbvLscore'] >= xbb_threshold(df_, ANALYSIS_XBB_WP)) & (df_['MET_pt'] >= 20) &
        (df_['ZH_M'] >= 50) & (df_['ZH_M'] <= 200) & (df_['ZH_pt'] >= 200)
    )

def _decay_from_dataset(name, proc):
    """1.0 = bb-only sample, 0.0 = non-bb sample, NaN = mixed/unknown (needs the event flag)."""
    n = name.lower()
    if proc == 'ttH':
        if 'nonbb' in n: return 0.0
        if 'tobb' in n:  return 1.0
    elif proc == 'ttZ':
        if 'tobb' in n:  return 1.0
        if any(k in n for k in ['llnunu', 'ttll', 'ttnunu']): return 0.0
    return np.nan   # e.g. TTZToQQ contains both Z->bb and Z->qq

def check_signal_decays(data_dict):
    print("\n" + "=" * 70)
    print(" NON-BB SIGNAL CHECK")
    print("=" * 70)

    for proc in sig_processes:
        df = data_dict.get(proc)
        if df is None or df.empty or 'dataset' not in df.columns:
            print(f" {proc}: no events or no dataset info, skipping")
            continue

        flag = SIGNAL_TRUTH_FLAG[proc]
        from_name = df['dataset'].map(lambda d: _decay_from_dataset(d, proc)).astype(float)

        if flag in df.columns:
            from_flag = df[flag].astype(float)
            is_bb = from_flag.where(from_flag.notna(), from_name)

            # Sanity check: per-event flag vs. sample name
            both = from_name.notna() & from_flag.notna()
            mismatch = both & (from_name != from_flag)
            if mismatch.any():
                print(f" 🚨 {proc}: {mismatch.sum()} events where events_{flag} disagrees with the sample name:")
                print(df.loc[mismatch, 'dataset'].value_counts().to_string())
        else:
            print(f" [WARNING] events_{flag} not found for {proc}; using sample names only")
            is_bb = from_name

        df['is_bb'] = is_bb   # kept on the dataframe for later use

        # Double counting: a bb-only sample alongside an inclusive sample that also contains bb
        has_bb_sample = (from_name == 1.0).any()
        inclusive_bb = (from_name.isna() & (is_bb == 1.0))
        if has_bb_sample and inclusive_bb.any():
            print(f" 🚨 {proc}: bb events found in both a dedicated bb sample and an inclusive sample "
                  f"({', '.join(df.loc[inclusive_bb, 'dataset'].unique())}). Possible double counting!")

        for sel_name, sel in [('Validation selection', pd.Series(True, index=df.index)),
                              (f'Analysis selection (Xbb {ANALYSIS_XBB_WP})', analysis_cuts(df))]:
            sub = df[sel]
            cat = np.select([sub['is_bb'] == 1.0, sub['is_bb'] == 0.0], ['bb', 'non-bb'], 'unknown')
            table = (pd.DataFrame({'dataset': sub['dataset'].values, 'cat': cat,
                                   'w': sub['tot_weight'].values})
                     .pivot_table(index='dataset', columns='cat', values='w',
                                  aggfunc='sum', fill_value=0.0))
            for c in ['bb', 'non-bb', 'unknown']:
                if c not in table.columns: table[c] = 0.0
            table = table[['bb', 'non-bb', 'unknown']]
            table['raw events'] = sub['dataset'].value_counts().reindex(table.index).fillna(0).astype(int)

            tot = table[['bb', 'non-bb', 'unknown']].sum()
            total = tot.sum()
            print(f"\n --- {proc}: {sel_name} (weighted yields) ---")
            print(table.round(3).to_string())
            if total != 0:
                print(f" Total: bb = {tot['bb']:.3f}, non-bb = {tot['non-bb']:.3f} "
                      f"({tot['non-bb'] / total:.2%}), unknown = {tot['unknown']:.3f} "
                      f"({tot['unknown'] / total:.2%})")
    print("=" * 70 + "\n")
    
def load_and_cut_data(variation='nominal', coffea_dir=COFFEA_DIR, year=TARGET_ERA):
    tracked_data = {proc: {} for proc in all_processes}

    # coffea_dir may be a single path or a list of paths (e.g. PreVFP + PostVFP)
    dirs = [coffea_dir] if isinstance(coffea_dir, str) else list(dict.fromkeys(coffea_dir))
    all_files = sorted({os.path.realpath(f)
                        for d in dirs
                        for f in glob.glob(os.path.join(d, "*/*.coffea"))})
    if not all_files:
        print(f"🚨 WARNING: no .coffea files found in {dirs}")
    
    file_mapping = {
        'ele_scale': 'shape_electron_scale_and_smearing',
        'ele_smear': 'shape_electron_scale_and_smearing',
        'muon_scale': 'shape_muons_scale_and_resolution',
        'muon_smear': 'shape_muons_scale_and_resolution',
        'unclust_En': 'shape_met_type1_calibration'
    }
    
    if variation == 'nominal':
        valid_files = [f for f in all_files if 'nom' in f.lower()]
    else:
        if variation.endswith('Up'):
            base_syst = variation[:-2]
        elif variation.endswith('Down'):
            base_syst = variation[:-4]
        else:
            base_syst = variation
        
        if 'AK4PFPuppi_' in base_syst or 'AK8PFPuppi_' in base_syst:
            search_string = base_syst.replace('AK4PFPuppi_', 'jec_').replace('AK8PFPuppi_', 'jec_')
        else:
            search_string = file_mapping.get(base_syst, base_syst)
            
        valid_files = [f for f in all_files if search_string.lower() in f.lower()]
        
        if len(valid_files) == 0:
            valid_files = [f for f in all_files if 'nom' in f.lower()]
        
    print(f"Extracting '{variation}' from {len(valid_files)} matching files...")

    warned_unknown_era = set()

    for file_path in valid_files:
        filename = os.path.basename(file_path)
        filein = load(file_path)
        genweight_dict = filein.get('sum_signOf_genweights', {})
        
        for raw_proc in filein['columns'].keys():
            mapped_proc = None
            if 'TTTo' in raw_proc: mapped_proc = 'TTBar' if 'tt+B' not in raw_proc else None
            elif 'TTbb' in raw_proc: mapped_proc = 'tt_B' if 'tt+B' in raw_proc else None
            elif 'WJets' in raw_proc or 'DYJets' in raw_proc: mapped_proc = 'VJets'
            elif 'QCD' in raw_proc: mapped_proc = 'QCD'
            elif 'ttH' in raw_proc or 'tth' in raw_proc.lower(): mapped_proc = 'ttH'
            elif 'TTZ' in raw_proc or 'ttz' in raw_proc.lower(): mapped_proc = 'ttZ'
            elif 'TTLL' in raw_proc or 'TTNuNu' in raw_proc: mapped_proc = 'ttZ'
            elif 'DATA' in raw_proc or 'data' in raw_proc.lower(): mapped_proc = 'data_obs'
            elif 'SingleTop' in raw_proc: mapped_proc = 'SingleTop'
            elif 'ttX' in raw_proc: mapped_proc = 'TTX'
            elif 'VV' in raw_proc: mapped_proc = 'VV'
            
            if mapped_proc not in all_processes: continue
                
            for dataset in filein['columns'][raw_proc].keys():

                # ==========================================================
                # ERA FILTERING (supports combined 2016 = PreVFP + PostVFP)
                # ==========================================================
                ds_era = get_dataset_era(dataset)
                if ds_era is None:
                    if dataset not in warned_unknown_era:
                        print(f"  -> [WARNING] Could not determine era of '{dataset}'; skipping it.")
                        warned_unknown_era.add(dataset)
                    continue
                if ds_era not in TARGET_SUB_ERAS:
                    continue
                # ==========================================================
                
                # --- NATIVE GENWEIGHT EXTRACTION ---
                genweight = genweight_dict.get(dataset, 1.0) 
                if isinstance(genweight, dict):
                    genweight = genweight.get(dataset, 1.0)
                        
                if genweight == 1.0 and mapped_proc != 'data_obs':
                    print(f"  -> [WARNING] Nominal genweight missing for {dataset}! Defaulting to 1.0.")
                
                try:
                    base_dict = filein['columns'][raw_proc][dataset]['btag_mask'][variation]
                    nom_dict = filein['columns'][raw_proc][dataset]['btag_mask']['nominal']
                except KeyError:
                    continue 
                    
                if mapped_proc == 'data_obs':
                    current_vars_to_extract = validation_vars + cut_vars
                    essential_vars = ['n_b_outZH', 'n_ak4jets'] + cut_vars
                else:
                    current_vars_to_extract = vars_to_extract + cut_vars
                    if mapped_proc in sig_processes:
                        current_vars_to_extract = current_vars_to_extract + list(SIGNAL_TRUTH_FLAG.values())
                    essential_vars = ['n_b_outZH', 'n_ak4jets', 'norm_weight'] + cut_vars
                
                tmp_data = {}
                skip_dataset = False
                
                for var in current_vars_to_extract:
                    dict_key = f'spanet_output_{var}' if var in ['ttzbb', 'tthbb', 'ttbb', 'ttlf', 'ttcc', 'signal'] else f'events_{var}'
                    
                    if dict_key in base_dict:
                        tmp_data[var] = np.array(base_dict[dict_key].value)
                    elif var in essential_vars:
                        if dict_key in nom_dict:
                            tmp_data[var] = np.array(nom_dict[dict_key].value)
                        else:
                            print(f"🚨 FATAL: Essential '{dict_key}' missing in {mapped_proc} ({dataset})!")
                            skip_dataset = True
                            break 
                    else:
                        continue 
                        
                if skip_dataset: continue 
                
                if tmp_data: 
                    if mapped_proc != 'data_obs':
                        # norm_weight from the processor is assumed to already contain
                        # xsec * lumi of this dataset's own sub-era
                        tmp_data['norm_weight'] = tmp_data['norm_weight'] / genweight

                    df = pd.DataFrame(tmp_data)
                    df['era'] = ds_era   # needed for per-era Xbb thresholds & b-tag normalisation

                    # b-tag shape SF normalization (BTV): r(era, sample, n_ak4jets) from btag_norm.py,
                    # applied to btag_sf and to each btag_sf<src>_up/_down with its own r
                    if mapped_proc != 'data_obs' and BTAG_NORM is not None:
                        df = apply_btag_norm(df, ds_era, raw_proc, BTAG_NORM)

                    if mapped_proc in sig_processes:
                        df['dataset'] = dataset
                    df = remove_ttzqq_zbb(df, dataset)
                    if dataset not in tracked_data[mapped_proc]:
                        tracked_data[mapped_proc][dataset] = []
                    tracked_data[mapped_proc][dataset].append(df)

    data_dict = {}
    for proc in all_processes:
        dfs_to_concat = []
        for dataset, branches in tracked_data[proc].items():
            dfs_to_concat.extend(branches)
            
        if dfs_to_concat:
            df = pd.concat(dfs_to_concat, ignore_index=True)
            df['era'] = df['era'].astype('category')

            # Event weight: includes btag_sf exactly once; btag_sf is already normalized
            # per era, sample and jet multiplicity by apply_btag_norm above
            if proc != 'data_obs':
                df['tot_weight'] = getZhbbWeight(df, year=year)
            else:
                df['tot_weight'] = 1.0
            
            df = df[cuts(df)] 
            data_dict[proc] = df
        else:
            data_dict[proc] = None
            
    return data_dict
    
# ==========================================
# 3. PRE-COMPUTE HISTOGRAMS TO SAVE RAM
# ==========================================
print("\n--- Starting Data Extraction & Histogramming ---")

hist_dict = {v: {p: {} for p in all_processes} for v in validation_vars}

for syst in systematics:
    current_data = load_and_cut_data(variation=syst, coffea_dir=COFFEA_DIR, year=TARGET_ERA)
    if syst == 'nominal':
        check_signal_decays(current_data)
    
    # ==================================================
    # DIAGNOSTIC: PRINT RAW UNWEIGHTED YIELDS
    # ==================================================
    if syst == 'nominal':
        print("\n" + "="*50)
        print(f" RAW UNWEIGHTED EVENT COUNTS (PASSING CUTS, {OUTPUT_TAG})")
        print("="*50)
        total_mc_raw = 0
        for proc in all_processes:
            df = current_data[proc]
            raw_count = len(df) if df is not None and not df.empty else 0
            line = f" {proc:<15} : {raw_count:,} events"
            if len(TARGET_SUB_ERAS) > 1 and raw_count > 0:
                per_era = df['era'].value_counts()
                line += "  (" + ", ".join(f"{e}: {per_era.get(e, 0):,}" for e in TARGET_SUB_ERAS) + ")"
            print(line)
            if proc != 'data_obs': 
                total_mc_raw += raw_count
        print("-" * 50)
        print(f" {'Total MC':<15} : {total_mc_raw:,} events")
        print("="*50 + "\n")
    # ==================================================
    
    for var in validation_vars:
        n_bins, x_min, x_max = binning_dict.get(var, default_binning)
        bins = np.linspace(x_min, x_max, n_bins + 1)
        
        for proc in all_processes:
            df = current_data[proc]
            if df is None: continue
                
            if df.empty:
                hist_dict[var][proc][syst] = {'counts': np.zeros(n_bins), 'err2': np.zeros(n_bins)}
                continue
                
            if var in df.columns:
                mask = df[var].notna()
                vals = np.clip(df[var][mask], None, bins[-1])
                weights = df['tot_weight'][mask].copy()
                
                if proc == 'tt_B': weights *= 1.318 # * 1.8 # k-factor
                
                # --- 1. Compute Shape/Nominal Histograms ---
                counts, _ = np.histogram(vals, bins=bins, weights=weights)
                err2, _ = np.histogram(vals, bins=bins, weights=weights**2)
                hist_dict[var][proc][syst] = {'counts': counts, 'err2': err2}
                
                # --- 2. Compute Weight Systematics (ONLY on nominal MC data) ---
                if syst == 'nominal' and proc != 'data_obs':
                    for w_syst, (nom_col, up_col, dn_col) in weight_syst_mapping.items():
                        up_w = weights.copy()
                        dn_w = weights.copy()
                        
                        if nom_col:
                            nom_vals = df.get(nom_col, pd.Series(1.0, index=df.index))[mask].fillna(1.0).values
                            safe_nom = np.where(nom_vals == 0, 1.0, nom_vals) 
                            
                            # --- Fallback to nominal values if UP/DOWN columns are missing ---
                            default_up_dn = pd.Series(nom_vals, index=df[mask].index)
                            up_vals = df.get(up_col, default_up_dn)[mask].fillna(1.0).values
                            dn_vals = df.get(dn_col, default_up_dn)[mask].fillna(1.0).values
                            
                            up_w *= (up_vals / safe_nom)
                            dn_w *= (dn_vals / safe_nom)
                        else:
                            up_vals = df.get(up_col, pd.Series(1.0, index=df.index))[mask].fillna(1.0).values
                            dn_vals = df.get(dn_col, pd.Series(1.0, index=df.index))[mask].fillna(1.0).values
                            up_w *= up_vals
                            dn_w *= dn_vals
                            
                        w_counts_up, _ = np.histogram(vals, bins=bins, weights=up_w)
                        w_err2_up, _ = np.histogram(vals, bins=bins, weights=up_w**2)
                        hist_dict[var][proc][f'{w_syst}Up'] = {'counts': w_counts_up, 'err2': w_err2_up}
                        
                        w_counts_dn, _ = np.histogram(vals, bins=bins, weights=dn_w)
                        w_err2_dn, _ = np.histogram(vals, bins=bins, weights=dn_w**2)
                        hist_dict[var][proc][f'{w_syst}Down'] = {'counts': w_counts_dn, 'err2': w_err2_dn}
            else:
                pass
                
    del current_data

# ==========================================
# 4. PLOTTING FUNCTION
# ==========================================
def plot_variables_to_pdf(var_names, hist_dictionary, syst_base_list, output_filename="Data_MC_Plots.pdf"):
    mc_processes = bkg_processes + sig_processes
    
    with PdfPages(output_filename) as pdf:
        for chunk_start in range(0, len(var_names), 9):
            chunk_vars = var_names[chunk_start : chunk_start + 9]
            
            fig = plt.figure(figsize=(24, 24))
            outer_grid = fig.add_gridspec(3, 3, wspace=0.3, hspace=0.3)
            
            for idx, var_name in enumerate(chunk_vars):
                row, col = idx // 3, idx % 3
                inner_grid = outer_grid[row, col].subgridspec(2, 1, height_ratios=[3, 1], hspace=0.00)
                ax = fig.add_subplot(inner_grid[0])
                rax = fig.add_subplot(inner_grid[1], sharex=ax)
                ax.tick_params(labelbottom=False)
                
                n_bins, x_min, x_max = binning_dict.get(var_name, default_binning)
                bins = np.linspace(x_min, x_max, n_bins + 1)
                bin_centers = 0.5 * (bins[1:] + bins[:-1])
                
                mc_hists, mc_labels, mc_colors_list = [], [], []
                total_mc_counts = np.zeros(n_bins)
                total_mc_stat_err2 = np.zeros(n_bins)
                total_mc_syst_err2 = np.zeros(n_bins)
                
                # Stack Nominal
                for proc in mc_processes:
                    nom_data = hist_dictionary[var_name][proc].get('nominal')
                    if not nom_data: continue
                        
                    counts, err2 = nom_data['counts'], nom_data['err2']
                    yield_total, stat_unc = np.sum(counts), np.sqrt(np.sum(err2))
                    
                    mc_hists.append(counts)
                    mc_labels.append(f"{process_labels.get(proc, proc)} ({yield_total:.1f} ± {stat_unc:.1f})")
                    mc_colors_list.append(mc_colors[proc])
                    
                    total_mc_counts += counts
                    total_mc_stat_err2 += err2

                # Calculate Systematics Envelope
                for s_base in syst_base_list:
                    syst_up_diff, syst_dn_diff = np.zeros(n_bins), np.zeros(n_bins)
                    for proc in mc_processes:
                        nom_data = hist_dictionary[var_name][proc].get('nominal')
                        if not nom_data:
                            continue
                        nom = nom_data['counts']
                        up = hist_dictionary[var_name][proc].get(f'{s_base}Up', {'counts': nom})['counts']
                        dn = hist_dictionary[var_name][proc].get(f'{s_base}Down', {'counts': nom})['counts']
                        syst_up_diff += (up - nom)
                        syst_dn_diff += (dn - nom)
                        
                    max_diff = np.max(np.abs(syst_up_diff))
                    nom_yield = np.sum(total_mc_counts)
                    if nom_yield > 0 and max_diff > (nom_yield * 0.1):
                        print(f"🚨 WARNING: '{s_base}' causing massive shift on '{var_name}'! Max bin diff: {max_diff:.1f}")

                    total_mc_syst_err2 += np.maximum(np.abs(syst_up_diff), np.abs(syst_dn_diff))**2

                # Total Uncertainty = Stat ⊕ Syst
                total_mc_err = np.sqrt(total_mc_stat_err2 + total_mc_syst_err2)

                if mc_hists:
                    hep.histplot(mc_hists, bins=bins, ax=ax, stack=True, histtype='fill', 
                                 label=mc_labels, color=mc_colors_list, sort='yield')

                ax.stairs(values=total_mc_counts + total_mc_err, 
                    baseline=np.clip(total_mc_counts - total_mc_err, 0, None),
                    edges=bins, fill=True, hatch='////', edgecolor='red', facecolor='none', 
                    label=r'Stat $\oplus$ Syst Unc.')

                # --- Process Data ---
                # Convert to float so we can use np.nan for blinding
                data_counts = hist_dictionary[var_name][data_process]['nominal']['counts'].astype(float)
                
                # BLINDING: Hide data for ZH_bbvLscore > 0.8
                if var_name == 'ZH_bbvLscore':
                    blind_mask = bin_centers > 0.8
                    data_counts[blind_mask] = np.nan
                
                data_err = np.sqrt(data_counts)
                data_yield = np.nansum(data_counts) # Use nansum to ignore the blinded bins
                
                if data_yield > 0:
                    data_lbl = f"Data ({data_yield:.0f} ± {np.sqrt(data_yield):.1f})"
                    hep.histplot(data_counts, bins=bins, ax=ax, stack=False, histtype='errorbar', 
                                 color='black', label=data_lbl, yerr=data_err)

                # --- Ratio Panel ---
                with np.errstate(divide='ignore', invalid='ignore'):
                    ratio = data_counts / total_mc_counts
                    
                    # Force absolute values to prevent Matplotlib ValueErrors 
                    # caused by negative MC sum-of-weights in the denominator
                    ratio_err = np.abs(data_err / total_mc_counts) 
                    mc_rel_err = np.abs(total_mc_err / total_mc_counts)

                for arr in [ratio, ratio_err, mc_rel_err]:
                    arr[np.isnan(arr) | np.isinf(arr)] = 0

                # Bounding the downward error: 
                # It must be >= 0, and it cannot be larger than the ratio itself (to prevent crossing 0)
                yerr_down = np.clip(ratio_err, 0, np.maximum(ratio, 0))

                rax.stairs(values=1 + mc_rel_err, 
                           baseline=np.clip(1 - mc_rel_err, 0, None),
                           edges=bins, fill=True, hatch='////', edgecolor='red', facecolor='none')
                           
                rax.errorbar(bin_centers, ratio, yerr=[yerr_down, ratio_err], fmt='ko', markersize=3)
                rax.axhline(1, color='black', linestyle='--')
                
                # --- Styling ---
                ax.set_ylabel("Events")
                ax.legend(loc='upper right', ncol=2, fontsize=10) 
                
                max_val = max(np.max(total_mc_counts), np.nanmax(data_counts) if np.any(~np.isnan(data_counts)) else 0)
                ax.set_ylim(0.1, max_val * 100 if max_val > 0 else 100)
                ax.set_yscale('log')
                
                rax.set_xlabel(var_name)
                rax.set_ylabel("Data / MC")
                rax.set_ylim(0, 2.5)
                
                hep.cms.label("Preliminary", data=(data_yield > 0), lumi=LUMI, ax=ax, com=13.0, fontsize=12)

            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig) 
            
    print(f"\nFinished generating plots. Saved to {output_filename}")
    
def plot_pileup_variations(hist_dictionary, output_filename="Pileup_Variations.pdf"):
    print(f"\n--- Generating Pileup Overlay Plots ---")
    mc_processes = bkg_processes + sig_processes
    vars_to_plot = ['nPV', 'nPVGood']
    
    with PdfPages(output_filename) as pdf:
        for var_name in vars_to_plot:
            if var_name not in hist_dictionary: 
                continue
            
            n_bins, x_min, x_max = binning_dict.get(var_name, default_binning)
            bins = np.linspace(x_min, x_max, n_bins + 1)
            bin_centers = 0.5 * (bins[1:] + bins[:-1])
            
            nom_counts = np.zeros(n_bins)
            up_counts = np.zeros(n_bins)
            dn_counts = np.zeros(n_bins)
            
            # Sum up all MC processes
            for proc in mc_processes:
                if 'nominal' in hist_dictionary[var_name][proc]:
                    nom_counts += hist_dictionary[var_name][proc]['nominal']['counts']
                if 'puWeightUp' in hist_dictionary[var_name][proc]:
                    up_counts += hist_dictionary[var_name][proc]['puWeightUp']['counts']
                if 'puWeightDown' in hist_dictionary[var_name][proc]:
                    dn_counts += hist_dictionary[var_name][proc]['puWeightDown']['counts']
            
            fig = plt.figure(figsize=(10, 10))
            gs = fig.add_gridspec(2, 1, height_ratios=[3, 1], hspace=0.05)
            ax = fig.add_subplot(gs[0])
            rax = fig.add_subplot(gs[1], sharex=ax)
            ax.tick_params(labelbottom=False)
            
            # --- Main Plot ---
            hep.histplot(nom_counts, bins=bins, ax=ax, label=f"Nominal (Yield: {np.sum(nom_counts):.1f})", color='black', histtype='step', linewidth=2)
            hep.histplot(up_counts, bins=bins, ax=ax, label=f"PU Up (Yield: {np.sum(up_counts):.1f})", color='red', histtype='step', linewidth=2)
            hep.histplot(dn_counts, bins=bins, ax=ax, label=f"PU Down (Yield: {np.sum(dn_counts):.1f})", color='blue', histtype='step', linewidth=2)
            
            ax.set_ylabel("Events", fontsize=16)
            ax.legend(loc='upper right', fontsize=14)
            hep.cms.label("Preliminary", data=False, lumi=LUMI, ax=ax, com=13.0, fontsize=16)
            
            # --- Ratio Plot ---
            with np.errstate(divide='ignore', invalid='ignore'):
                ratio_up = np.where(nom_counts > 0, up_counts / nom_counts, 0)
                ratio_dn = np.where(nom_counts > 0, dn_counts / nom_counts, 0)
            
            rax.plot(bin_centers, ratio_up, 'r.-', label="Up / Nom", markersize=8)
            rax.plot(bin_centers, ratio_dn, 'b.-', label="Down / Nom", markersize=8)
            rax.axhline(1, color='black', linestyle='--')
            
            rax.set_xlabel(var_name, fontsize=16)
            rax.set_ylabel("Var / Nom", fontsize=14)
            rax.set_ylim(0.5, 1.5)
            rax.legend(loc='upper right', fontsize=10, ncol=2)
            
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)
            
    print(f"Saved pileup comparison plots to {output_filename}")

def optimize_tagger_wp(hist_dictionary, tagger_var='ZH_bbvLscore', sig_procs=['ttZ', 'ttH'], bkg_procs=['VJets', 'QCD', 'tt_B', 'TTBar', 'SingleTop', 'TTX']):
    print(f"\n--- Optimizing Working Point for {tagger_var} ---")
    
    if tagger_var not in hist_dictionary:
        print(f"Error: {tagger_var} not found in pre-computed histograms.")
        return

    n_bins, x_min, x_max = binning_dict.get(tagger_var, default_binning)
    bins = np.linspace(x_min, x_max, n_bins + 1)
    
    nom_sig_counts = np.zeros(n_bins)
    nom_bkg_counts = np.zeros(n_bins)
    
    for proc in sig_procs:
        if 'nominal' in hist_dictionary[tagger_var][proc]:
            nom_sig_counts += hist_dictionary[tagger_var][proc]['nominal']['counts']
            
    for proc in bkg_procs:
        if 'nominal' in hist_dictionary[tagger_var][proc]:
            nom_bkg_counts += hist_dictionary[tagger_var][proc]['nominal']['counts']

    # Calculate cumulative yields right-to-left (yield passing cut > bin_left_edge)
    sig_passing = np.cumsum(nom_sig_counts[::-1])[::-1]
    bkg_passing = np.cumsum(nom_bkg_counts[::-1])[::-1]
    
    # S / sqrt(B)
    with np.errstate(divide='ignore', invalid='ignore'):
        s_sqrt_b = np.where(bkg_passing > 0, sig_passing / np.sqrt(bkg_passing), 0)

    best_idx = np.nanargmax(s_sqrt_b)
    best_threshold = bins[best_idx]
    best_s_sqrt_b = s_sqrt_b[best_idx]
    
    total_sig = sig_passing[0]
    total_bkg = bkg_passing[0]
    
    opt_sig_eff = sig_passing[best_idx] / total_sig if total_sig > 0 else 0
    opt_mistag = bkg_passing[best_idx] / total_bkg if total_bkg > 0 else 0
    
    print(f"Optimal Threshold: > {best_threshold:.3f}")
    print(f"Max S/sqrt(B):     {best_s_sqrt_b:.3f}")
    print(f"Signal Efficiency: {opt_sig_eff:.2%}")
    print(f"Overall Bkg Mistag: {opt_mistag:.2%}")
    print(f"Background Yield:  {bkg_passing[best_idx]:.1f} events remaining")
    
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(bins[:-1], s_sqrt_b, 'k-', lw=2, label=r'$S/\sqrt{B}$')
    ax.axvline(best_threshold, color='r', linestyle='--', label=f'Optimal WP: > {best_threshold:.2f}')
    
    ax.set_xlabel(f"{tagger_var} Cut Threshold")
    ax.set_ylabel(r"$S/\sqrt{B}$")
    ax.set_title(f"Tagger Optimization in Analysis Phase Space")
    ax.legend(loc='upper left')
    hep.cms.label("Preliminary", data=False, lumi=LUMI, ax=ax)
    
    plt.tight_layout()
    plt.savefig(f"WP_Optimization_{tagger_var}_{OUTPUT_TAG}.pdf")
    plt.close()
    
    return best_threshold

# ==========================================
# 5. EXECUTE
# ==========================================
all_syst_bases = syst_bases + list(weight_syst_mapping.keys())

#plot_pileup_variations(hist_dict, f"Pileup_Variations_{OUTPUT_TAG}.pdf")
#optimal_cut = optimize_tagger_wp(hist_dict, tagger_var='ZH_bbvLscore')

plot_variables_to_pdf(validation_vars, hist_dict, all_syst_bases, f"Data_MC_WeightSys_{OUTPUT_TAG}.pdf")

 Era        : 2018  (sub-eras: 2018)
 Lumi       : 59.8 fb^-1
 Input dirs : Nominal2018
 Xbb cut    : none
 b-tag norm : btag_calib/btag_norm_sig.json, btag_calib/btag_norm_bkg.json, btag_calib/btag_norm_minor.json
 Output tag : 2018_NoXbb

--- Starting Data Extraction & Histogramming ---
Extracting 'nominal' from 4 matching files...
🚨 FATAL: Essential 'events_norm_weight' missing in ttH (ttHTobb_2018)!
🚨 FATAL: Essential 'events_norm_weight' missing in ttH (ttHTobb_2018)!
  -> TTZToQQ_2018: removed 129,301 of 129,301 Z->bb events
  -> TTZToQQ_2018: removed 263,572 of 1,126,062 Z->bb events

 NON-BB SIGNAL CHECK

 --- ttZ: Validation selection (weighted yields) ---
cat                    bb   non-bb  unknown  raw events
dataset                                                
TTZToBB_2018      103.123    0.000      0.0      177787
TTZToLLNuNu_2018    0.000   57.207      0.0       87994
TTZToQQ_2018        0.000  194.032      0.0      442303
 Total: bb = 103.123, non-bb = 251.239 (70.90%